This module aims to bulk download original reports from urls scraped and cleansed. Special notices include:
1. Set "headers" in GET requests as shown in your Browser (mimicking ordinary file download)
2. Set time lag/pause between requests and add randomness (imitating human behavior of file download)
3. Make sure VPN/proxy server is switched on when running the code



In [ ]:
import os
import time
from datetime import datetime
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import pandas as pd
import random

# Check output_folder and continent_selected before running the code
continent_selected = "africa" # africa/asia/europe/northamerica/southamerica
output_folder = "../../data/original_report/" + ("usgs_" + continent_selected) + "/"

# Read-in merged URL list
url_path = "../../data/merged_url/usgs_merged.csv"
url_table = pd.read_csv(url_path)
url_table['year'] = pd.to_datetime(url_table['year'])

# url_list is a copy or slice of url_table (based on continent_selected)
url_list = url_table.query("continent == @continent_selected and index >= 198") # IMPORTANT: Due to unexpected sleep of the system, "and index >= 198" is temporarily attached to continue from where the process terminated.

# Re-check
print(output_folder)
url_list

../../data/original_report/usgs_africa/


,year,country,url,continent
198,2005-01-01,Eswatini (formerly Swaziland),https://d9-wret.s3.us-west-2.amazonaws.com/ass...,africa
199,2014-01-01,Eswatini (formerly Swaziland),https://d9-wret.s3.us-west-2.amazonaws.com/ass...,africa
200,2015-01-01,Eswatini (formerly Swaziland),https://d9-wret.s3.us-west-2.amazonaws.com/ass...,africa
201,2016-01-01,Eswatini (formerly Swaziland),https://d9-wret.s3.us-west-2.amazonaws.com/ass...,africa
202,2018-01-01,Eswatini (formerly Swaziland),https://d9-wret.s3.us-west-2.amazonaws.com/ass...,africa
...,...,...,...,...
750,2014-01-01,Zimbabwe,https://d9-wret.s3.us-west-2.amazonaws.com/pal...,africa
751,2015-01-01,Zimbabwe,https://d9-wret.s3.us-west-2.amazonaws.com/pal...,africa
752,2016-01-01,Zimbabwe,https://d9-wret.s3-us-west-2.amazonaws.com/ass...,africa
753,2018-01-01,Zimbabwe,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,africa


In [14]:
## Define environment for request.Session()
# Define headers
headers_strategy = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0'
}

# Define retry strategy
retry_strategy = Retry(
    total = 5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)
retry_strategy_adapter = HTTPAdapter(max_retries=retry_strategy)

## Initialize a log file
log_folder = "../../log/" # Folder where all files are stored
log_name = 'downloadfile_' + str(int(time.time())) + '.txt' # Both name of task and timestamp included
log_path = os.path.join(log_folder, log_name)
# Open the log before webspider begins to operate
with open(log_path, 'w', encoding='utf-8') as log: # IMPORTANT: Encoding is designated as 'utf-8' given error returned in writing "Eswatini (formerly Swaziland)"-related entries (by default 'gbk' encoding). 
    
    ## WEBSPIER SECTION
    with requests.Session() as session:
        # Update environment of the requests.Session() with parameters already defined
        session.headers.update(headers_strategy)
        session.mount(prefix='https://', adapter=retry_strategy_adapter)
        session.mount(prefix='http://', adapter=retry_strategy_adapter)

        counter = 0 # Initiate a counter for indicating download process
        for index in url_list.index: # Iterate through url_list
            counter = counter + 1 # Update the counter

            # Extract basic information about file to download from the row corresponding to the index
            year = str(url_list.loc[index]['year'].year)
            country = url_list.loc[index]['country']
            continent = url_list.loc[index]['continent']
            url = url_list.loc[index]['url']
            # Name the file to download and save ("[country]_[year].pdf")
            filename = '_'.join([country,year])+'.pdf'
            filepath = os.path.join(output_folder, filename)
            
            # Start the timer 
            start_time = time.perf_counter()
            start_time_legible = str(datetime.now())

            try:
                # Print and write into log the beginning of operation
                message_start = f"Downloading {counter}/{len(url_list.index)} [{country} in {year} from: {url}]"
                print(message_start)
                log.write(start_time_legible + '\n')
                log.write(message_start + '\n\n') # Add two line breaks

                ## CORE OF WEBSPIDER
                # Get request ()
                response = session.get(
                    url = url,
                    timeout = (5,30), # 5s connect timeout limit and 30s read timeout limit
                    stream = True # Temporarily save response to hard drive rather than RAM and prepare for iteration below
                )
                # Check whether the request is properly handled by the server (jump down to "except" if status_code is not 200 or there's a timeout)
                response.raise_for_status()
                # Write response into pdf-suffix file by chunk (chunk_size set as 8192 bytes or 8 KB based on convention)
                with open(filepath,'wb') as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        file.write(chunk)

                # End the timer
                end_time = time.perf_counter()
                end_time_legible = str(datetime.now())
                execution_time = end_time - start_time # Calculate execution interval
                # Print and write to log the termination of operation
                message_end = f"Successfully saved to {filepath}. Execution time {execution_time:.2f} seconds"
                print(message_end)
                log.write(end_time_legible + '\n')
                log.write(message_end + '\n\n') # Add two line breaks

                ## CORE OF WEBSPIDER (CONTINUE): pause for random seconds (no more than 4s) before initiate another request 
                time.sleep(2 + random.random() * 2)
            
            except requests.exceptions.RequestException as error:
                # End the timer
                end_time = time.perf_counter()
                end_time_legible = str(datetime.now())
                execution_time = end_time - start_time
                # Print and write to log the termination of operation
                message_end = f"Failed to download {url}. Error: {error}. Execution time {execution_time:.2f} seconds"
                print(message_end)
                log.write(end_time_legible + '\n')
                log.write(message_end + '\n\n')

                ## CORE OF WEBSPIDER (CONTINUE): pause for random seconds (no more than 7s) before initiate another request 
                time.sleep(5 + random.random() * 2)

Successfully saved to ../../data/original_report/usgs_africa/Eswatini (formerly Swaziland)_2005.pdf. Execution time 1.46 seconds
Successfully saved to ../../data/original_report/usgs_africa/Eswatini (formerly Swaziland)_2014.pdf. Execution time 0.56 seconds
Successfully saved to ../../data/original_report/usgs_africa/Eswatini (formerly Swaziland)_2015.pdf. Execution time 0.34 seconds
Successfully saved to ../../data/original_report/usgs_africa/Eswatini (formerly Swaziland)_2016.pdf. Execution time 0.35 seconds
Successfully saved to ../../data/original_report/usgs_africa/Eswatini (formerly Swaziland)_2018.pdf. Execution time 0.31 seconds
Successfully saved to ../../data/original_report/usgs_africa/Eswatini (formerly Swaziland)_2019.pdf. Execution time 2.65 seconds
Successfully saved to ../../data/original_report/usgs_africa/Ethiopia_2005.pdf. Execution time 1.58 seconds
Successfully saved to ../../data/original_report/usgs_africa/Ethiopia_2006.pdf. Execution time 0.36 seconds
Successful